# RP-PCA Asset Pricing Model
**Implementation of:** *Estimating Latent Asset-Pricing Factors* — Lettau & Pelger (2018)

This notebook implements Risk-Premium PCA (RP-PCA) on anomaly-sorted decile portfolio returns to extract latent pricing factors and construct an investable equity portfolio.

**Pipeline:**
1. Load anomaly-sorted portfolio returns from [Open Source Asset Pricing](https://www.openassetpricing.com/data/)
2. Fit RP-PCA with automatic factor-count selection (eigenvalue ratio test)
3. Evaluate in-sample and out-of-sample Sharpe ratios and pricing errors
4. Map factor SDF weights to individual NYSE/NASDAQ stocks via market-cap weighting

**Paper:** Lettau, M. & Pelger, M. (2020). Factors That Fit the Time Series and Cross-Section of Stock Returns. *The Review of Financial Studies*, 33(5), 2274–2325.

## Quick-Start Guide

**First time setup or after a new data release:**
> Run **Section 1** once to download and convert the raw data, then continue to Section 2.

**Normal use (data already in Drive):**
> Skip Section 1. Start at **Section 2** and run all cells in order.

All tunable parameters are consolidated in the **Configuration** cell directly below.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — Edit this cell, then run all cells below
# ═══════════════════════════════════════════════════════════════

# ── Data Paths ──────────────────────────────────────────────────
DRIVE_DATA_DIR  = '/content/drive/MyDrive/rp-pca-data'  # Google Drive folder holding the parquet data files
MKT_CAP_CACHE   = f'{DRIVE_DATA_DIR}/mkt_cap_map.json'

# ── Data Source URL (Section 1 updates only) ────────────────────
DATA_FOLDER_URL = 'https://drive.google.com/drive/folders/1qQDuTsnyvWfEJR6nPBQZ8xxlq6bkLG_y?usp=drive_link'

# ── Sample Period ────────────────────────────────────────────────
START_DATE = '1960-01-01'

# ── Signals to Exclude ──────────────────────────────────────────
# Run the correlation helper in Section 2 to identify new candidates.
EXCLUDE_SIGNALS = ['PriceDelayTstat', 'grcapx3y', 'zerotrade1M', 'zerotrade6M', 'BM']

# ── RP-PCA Parameters ────────────────────────────────────────────
GAMMA         = 10.0  # Risk-premium weight (γ). Paper recommends 10.
K_MANUAL      = 5     # Number of factors (paper's benchmark analyses use K of 3-6).
USE_AUTO_K    = False # True = let the eigenvalue-ratio test choose K instead of K_MANUAL.
K_MAX         = 10    # Upper bound for the background eigenvalue-ratio diagnostic.
OOS_WINDOW    = 240   # Rolling OOS window in months (240 = 20 years).
STD_NORM      = 0     # Vol-standardize returns before PCA? (0 = no, paper default)
VAR_NORM      = 1     # Scale loadings by sqrt(eigenvalues)? (1 = yes, paper default)
ORTHOGONALIZE = 1     # Orthogonalize factors via QR? (1 = yes, paper default)

# ── Decile Selection ─────────────────────────────────────────────
# 'extreme' → deciles 1 & 10 only (paper benchmark, N ≈ 2 × num_signals)
# 'all'     → all 10 deciles (more information, N ≈ 10 × num_signals, slower)
DECILE_MODE = 'extreme'

# ── Universe Filters ─────────────────────────────────────────────
MIN_DATE_YYYYMM = 200000  # Use characteristic data no older than year 2000.
STALENESS_DAYS  = 15      # Drop stocks not traded within this many days.
MKT_CAP_BATCH   = 40      # Tickers per yfinance batch (keep ≤ 40).


---
# Section 1 — Data Update
### ⚠️  Only run this section when a new data release has been published.

To check for updates, visit: https://www.openassetpricing.com/data/

If new data is available:
1. Copy the new Google Drive folder link into `DATA_FOLDER_URL` in the Configuration cell above.
2. Run the cell below. It will download, convert, and store the updated parquet files in your Shared Drive.
3. Proceed to Section 2.

In [ ]:
import os, shutil, zipfile, re
import gdown
import polars as pl
from google.colab import auth, drive
from googleapiclient.discovery import build

# Files to download (path within the public Drive folder → local target name)
FILE_TARGETS = [
    ['Firm Level Characteristics', 'Full Sets', 'signed_predictors_dl_wide.zip'],
    ['Portfolios', 'Full Sets Alt', 'PredictorAltPorts_DecilesVW.zip'],
]

def _mount_drive():
    if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
        shutil.rmtree('/content/drive')
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive', force_remount=True)

def _extract_folder_id(url):
    m = re.search(r'folders/([a-zA-Z0-9_-]+)', url)
    if not m:
        raise ValueError(f'Could not parse folder ID from URL: {url}')
    return m.group(1)

def _get_child_id(service, parent_id, name):
    q = f"'{parent_id}' in parents and name = '{name}' and trashed = false"
    files = service.files().list(q=q, fields='files(id,name)').execute().get('files', [])
    if not files:
        raise FileNotFoundError(f"'{name}' not found in Drive folder.")
    return files[0]['id']

def update_data(folder_url=DATA_FOLDER_URL, target_dir=DRIVE_DATA_DIR):
    """Download the latest data from Open Source Asset Pricing and store as parquet."""
    auth.authenticate_user()
    service = build('drive', 'v3')
    _mount_drive()

    root_id = _extract_folder_id(folder_url)
    tmp = '/content/_oapdata_tmp'

    # Clean up previous run artifacts
    if os.path.exists(target_dir):
        print(f'Clearing old data in {target_dir}...')
        shutil.rmtree(target_dir)
    os.makedirs(target_dir, exist_ok=True)
    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    os.makedirs(tmp)

    # Download, unzip, and convert each file
    for path_parts in FILE_TARGETS:
        zip_name = path_parts[-1]
        try:
            current_id = root_id
            for part in path_parts:
                current_id = _get_child_id(service, current_id, part)
            local_zip = f'/content/{zip_name}'
            gdown.download(f'https://drive.google.com/uc?id={current_id}', local_zip, quiet=False)
            with zipfile.ZipFile(local_zip, 'r') as z:
                z.extractall(tmp)
            os.remove(local_zip)
            print(f'  ✅ Downloaded and extracted: {zip_name}')
        except Exception as e:
            print(f'  ❌ Failed on {zip_name}: {e}')

    # Convert CSV files to parquet (streaming to handle large files)
    print('Converting to parquet...')
    for fname in os.listdir(tmp):
        if fname.endswith('.csv'):
            src = os.path.join(tmp, fname)
            dst = os.path.join(target_dir, fname.replace('.csv', '.parquet'))
            lf = pl.scan_csv(src, infer_schema_length=10_000,
                             schema_overrides={'port': pl.String})
            lf.sink_parquet(dst)
            print(f'  Converted {fname}')
        elif fname.endswith('.parquet'):
            shutil.move(os.path.join(tmp, fname), os.path.join(target_dir, fname))

    shutil.rmtree(tmp)
    print('\n✅ Data update complete. You can now run Section 2.')

update_data()


---
# Section 2 — Data Loading & Signal Selection

Mounts Google Drive, loads the anomaly portfolio returns and firm characteristics as lazy Polars frames, and selects the signals used in the RP-PCA.

In [ ]:
import os, json, time, gc, logging, warnings, shutil
import numpy as np
import polars as pl
import pandas as pd
from scipy import linalg
import yfinance as yf
import tqdm
import matplotlib.pyplot as plt
from google.colab import drive

warnings.filterwarnings('ignore')
logging.getLogger('yfinance').setLevel(logging.CRITICAL)

# ── Mount Google Drive ───────────────────────────────────────────
if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive')
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print('Drive already mounted.')

# ── Load Data as Lazy Frames ─────────────────────────────────────
# scan_parquet keeps data on disk; nothing is loaded into memory yet.
predictors = pl.scan_parquet(f'{DRIVE_DATA_DIR}/signed_predictors_dl_wide.parquet')
portfolios  = pl.scan_parquet(f'{DRIVE_DATA_DIR}/PredictorAltPorts_DecilesVW.parquet')

all_signals = (
    portfolios.select('signalname').unique().collect()
    .to_series().sort().to_list()
)
print(f'✅ Loaded. {len(all_signals)} signals available.')


In [ ]:
# ── Select Signals ──────────────────────────────────────────────
selected_signals = [s for s in all_signals if s not in EXCLUDE_SIGNALS]
print(f'Using {len(selected_signals)} of {len(all_signals)} signals '
      f'({len(EXCLUDE_SIGNALS)} excluded).')

# ── Optional: Identify Highly-Correlated Signal Pairs ───────────
# Useful for deciding which signals to add to EXCLUDE_SIGNALS.
# Uncomment the block below to run.

def find_high_correlations(portfolios_df, signals, port='10', threshold=0.85):
    """
    Return pairs of signals whose top-decile returns are correlated
    above `threshold`. High correlation indicates redundant signals.

    Parameters
    ----------
    portfolios_df : Polars LazyFrame of portfolio returns
    signals       : list of signal names to check
    port          : decile to use for correlation ('10' = top decile)
    threshold     : absolute correlation cutoff
    """
    df = (
        portfolios_df
        .filter(pl.col('signalname').is_in(signals) & (pl.col('port') == port))
        .collect()
        .pivot(values='ret', index='date', on='signalname')
        .fill_null(0)
    )
    corr = df.select(pl.exclude('date')).to_pandas().corr()
    cols = corr.columns.tolist()
    pairs = [
        (cols[i], cols[j], round(corr.iloc[i, j], 3))
        for i in range(len(cols))
        for j in range(i + 1, len(cols))
        if abs(corr.iloc[i, j]) >= threshold
    ]
    return sorted(pairs, key=lambda x: -abs(x[2]))

# high_corr = find_high_correlations(portfolios, selected_signals)
# print(f'Found {len(high_corr)} highly correlated pairs (≥{0.85}):')
# for a, b, r in high_corr[:20]:
#     print(f'  {a} <-> {b}: r={r:.3f}')


---
# Section 3 — RP-PCA Factor Analysis

Defines the model functions, chooses the number of factors K (manual by default; an eigenvalue-ratio diagnostic runs in the background), then runs the full in-sample and rolling out-of-sample analysis.

In [ ]:
def rppca(X, gamma, K, stdnorm=0, variance_normalization=1, orthogonalization=1):
    """
    Risk-Premium PCA estimator (Lettau & Pelger 2020).

    RP-PCA augments standard PCA by adding a risk-premium penalty term
    (controlled by gamma) that tilts extracted factors toward the cross-section
    of expected returns. This makes it far better at detecting 'weak' factors
    that have high Sharpe ratios but low variance.

    The modified covariance matrix is:
        V = X' (I + gamma * 11'/T) X / T
    Eigenvectors of V are the RP-PCA loadings.

    Parameters
    ----------
    X                      : (T, N) numpy array of excess returns
    gamma                  : float, risk-premium weight (>=0; 0 = standard PCA)
    K                      : int, number of factors to extract
    stdnorm                : int, 1 = standardize each asset to unit volatility first
    variance_normalization : int, 1 = scale loadings by sqrt(eigenvalues)
    orthogonalization      : int, 1 = orthogonalize factors via QR decomposition

    Returns
    -------
    dict with keys:
        'loadings'         : (N, K) factor loadings (Lambda-hat)
        'factors'          : (T, K) estimated factors (F-hat)
        'eigenvalues'      : (N,) eigenvalues of RP-PCA covariance, sorted descending
        'SDF'              : (T, K) stochastic discount factor for k=1..K factors
        'SDFweightsassets' : dict {k: (N,) asset-space SDF weights; X @ w == SDF[:, k-1]}
        'beta'             : dict {k: (N, k) factor betas}
        'alpha'            : (N, K) pricing errors (intercepts)
        'residual'         : dict {k: (T, N) OLS residuals}
        'factorweight'     : (N, K) normalized asset weights for each factor
    """
    if hasattr(X, 'to_numpy'):
        X = X.to_numpy()
    T, N = X.shape

    # Optional cross-sectional vol standardization
    if stdnorm == 1:
        X_centered = X - X.mean(axis=0)
        vol = np.sqrt(np.diag(X_centered.T @ X_centered / T))
        WN = np.diag(1.0 / vol)
    else:
        WN = np.eye(N)

    # RP-PCA covariance: standard PCA + risk-premium term
    WT = np.eye(T) + gamma * np.ones((T, T)) / T
    X_tilde = X @ WN
    V = X_tilde.T @ WT @ X_tilde / T

    # Eigendecomposition (sorted descending)
    eigenvalues, eigenvectors = linalg.eigh(V)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues  = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Revert vol-standardization to get loadings in original return units
    Lambda = linalg.inv(WN.T) @ eigenvectors[:, :K]

    # Sign normalization: each factor should on average be positive
    proj_mean = np.mean(X @ Lambda @ linalg.inv(Lambda.T @ Lambda), axis=0)
    Lambda = Lambda @ np.diag(np.sign(proj_mean))

    # Factor weights: columns of (Lambda (Lambda'Lambda)^-1), then col-normalized
    FW = Lambda @ linalg.inv(Lambda.T @ Lambda)
    FW = FW @ np.diag(1.0 / np.sqrt(np.diag(FW.T @ FW)))
    F  = X @ FW

    # Variance normalization and/or orthogonalization
    if variance_normalization == 1 and orthogonalization == 0:
        Lambda = Lambda @ np.diag(np.sqrt(eigenvalues[:K]))
        FW     = FW    @ np.diag(1.0 / np.sqrt(eigenvalues[:K]))
        F      = X @ FW

    elif orthogonalization == 1:
        F_c = (np.eye(T) - np.ones((T, T)) / T) @ F / np.sqrt(T)
        Q, R = linalg.qr(F_c)
        if variance_normalization == 1:
            Rot = linalg.inv(R[:K, :K])
            FW  = FW @ Rot
        else:
            # Rot is kept unmodified so Lambda can invert it correctly.
            # The additional per-column normalization applies only to FW.
            Rot   = linalg.inv(R[:K, :K]) @ np.diag(np.diag(R[:K, :K]))
            FW    = FW @ Rot
            norms = np.sqrt(np.diag(FW.T @ FW))
            FW    = FW @ np.diag(1.0 / norms)
        F      = X @ FW
        sign_d = np.diag(np.sign(F.mean(axis=0)))
        F      = F  @ sign_d
        FW     = FW @ sign_d
        Lambda = Lambda @ linalg.inv(Rot) @ sign_d

    # Build SDF and asset-space SDF weights for each k = 1 .. K
    SDF              = np.zeros((T, K))
    SDFweightsassets = {}
    alpha            = np.zeros((N, K))
    beta             = {}
    residual         = {}

    for k in range(1, K + 1):
        Lk = Lambda[:, :k]
        Fk = F[:, :k]

        # SDF weights solve: Cov(F) w = Mean(F)
        mu_f  = Fk.mean(axis=0)
        cov_f = np.atleast_2d(np.cov(Fk, rowvar=False))
        w_sdf = linalg.solve(cov_f, mu_f)

        SDF[:, k - 1]       = Fk @ w_sdf
        # Asset-space SDF weights. Since F = X @ FW by construction,
        # X @ (FW[:, :k] @ w_sdf) reproduces the SDF exactly under any
        # normalization/orthogonalization setting.
        SDFweightsassets[k] = FW[:, :k] @ w_sdf

        # OLS of returns on factors
        regressors = np.hstack([np.ones((T, 1)), Fk])
        coefs      = linalg.inv(regressors.T @ regressors) @ regressors.T @ X
        residual[k]        = X - regressors @ coefs
        alpha[:, k - 1]    = coefs[0, :]
        beta[k]            = coefs[1:, :].T

    return {
        'loadings':         Lambda,
        'factors':          F,
        'eigenvalues':      eigenvalues,
        'SDF':              SDF,
        'SDFweightsassets': SDFweightsassets,
        'beta':             beta,
        'alpha':            alpha,
        'residual':         residual,
        'factorweight':     FW,
    }


In [ ]:
def select_k(eigenvalues, X_shape, k_max=10):
    """
    Automatically select the number of RP-PCA factors K.

    Uses the Ahn-Horenstein (2013) eigenvalue ratio test:
        K* = argmax_{k=1..k_max}  lambda_k / lambda_{k+1}
    The largest ratio indicates the most meaningful drop-off in
    explained variation, signalling that additional factors are noise.

    As a secondary check, any factors whose eigenvalue lies below the
    Marchenko-Pastur bulk edge (i.e. consistent with pure noise under
    a random matrix) are flagged.

    Parameters
    ----------
    eigenvalues : array-like, sorted descending
    X_shape     : (T, N) of the returns matrix
    k_max       : maximum K to consider

    Note
    ----
    When one factor dominates the spectrum (e.g. a strong market factor),
    the ratio test tends to select K=1. If this happens, inspect the scree
    plot and consider setting USE_AUTO_K = False with a manual K_MANUAL
    (the paper's benchmark analyses use K around 3-6).

    Returns
    -------
    K : int
    """
    T, N = X_shape
    eigs = np.array(eigenvalues)[:k_max + 1]

    # Eigenvalue ratio test
    ratios = eigs[:-1] / (eigs[1:] + 1e-12)
    K = int(np.argmax(ratios)) + 1

    # Marchenko-Pastur upper edge as a sanity check
    sigma2   = np.median(eigenvalues)
    mp_edge  = sigma2 * (1 + np.sqrt(N / T)) ** 2
    n_above  = int(np.sum(eigs > mp_edge))
    if n_above != K:
        print(f'  ℹ️  Note: ratio test → K={K}, '
              f'Marchenko-Pastur test → K={n_above}. '
              f'Using ratio test result.')

    return max(1, min(K, k_max))


def plot_scree(eigenvalues, K, X_shape, k_max=15):
    """
    Plot the eigenvalue scree plot and ratio plot side by side,
    marking the selected K and the Marchenko-Pastur bulk edge.
    """
    T, N = X_shape
    eigs    = np.array(eigenvalues)[:k_max]
    k_range = np.arange(1, len(eigs) + 1)
    sigma2  = np.median(eigenvalues)
    mp_edge = sigma2 * (1 + np.sqrt(N / T)) ** 2

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.plot(k_range, eigs, 'o-', color='steelblue', lw=2)
    ax1.axhline(mp_edge, color='tomato', ls='--', label=f'MP bulk edge ({mp_edge:.4f})')
    ax1.axvline(K, color='seagreen', ls='--', lw=2, label=f'Selected K = {K}')
    ax1.set_xlabel('Factor k'); ax1.set_ylabel('Eigenvalue')
    ax1.set_title('Scree Plot'); ax1.legend()

    ratios = eigs[:-1] / (eigs[1:] + 1e-12)
    ax2.bar(k_range[:-1], ratios, color='steelblue', alpha=0.7)
    ax2.axvline(K, color='seagreen', ls='--', lw=2, label=f'Selected K = {K}')
    ax2.set_xlabel('Factor k'); ax2.set_ylabel(r'$\lambda_k / \lambda_{k+1}$')
    ax2.set_title('Eigenvalue Ratio (Ahn-Horenstein 2013)'); ax2.legend()

    plt.suptitle('RP-PCA Factor Selection', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
def prepare_data_for_rppca(portfolios_df, signals, start_date=START_DATE,
                           decile_mode=DECILE_MODE):
    """
    Transform PredictorAltPorts_DecilesVW into a balanced returns matrix
    suitable for RP-PCA.

    The paper uses only the extreme deciles (1 and 10) as test assets,
    which concentrates the pricing signal. Using all 10 deciles gives
    more data but is slower.

    Missing data handling:
      - Columns with >20% missing returns are dropped (signal likely
        unavailable for much of the sample).
      - Remaining gaps are filled with 0 (consistent with balanced-panel
        assumption in the paper).

    Parameters
    ----------
    portfolios_df : Polars LazyFrame
    signals       : list of signal names
    start_date    : ISO string, e.g. '1960-01-01'
    decile_mode   : 'extreme' or 'all'

    Returns
    -------
    X            : (T, N) numpy array of fractional returns
    dates        : Polars Series of dates
    asset_cols   : list of column names in X, e.g. ['Accruals_01', 'Accruals_10', ...]
    """
    y, m, d = [int(x) for x in start_date.split('-')]
    ports = ['01', '10'] if decile_mode == 'extreme' \
            else [f'{i:02d}' for i in range(1, 11)]

    df = (
        portfolios_df
        .filter(pl.col('date').cast(pl.Date) >= pl.date(y, m, d))
        .filter(pl.col('signalname').is_in(signals))
        .filter(pl.col('port').is_in(ports))
        .with_columns(
            (pl.col('signalname') + '_' + pl.col('port')).alias('asset_id')
        )
        .collect()
        .pivot(values='ret', index='date', on='asset_id')
        .sort('date')
    )

    dates = df.get_column('date')
    X_raw = df.select(pl.exclude('date')) / 100.0  # percentage → decimal

    # Drop columns with excessive missing data
    null_fracs = X_raw.null_count().row(0)  # one row: null count per column
    keep = [
        col for col, n_null in zip(X_raw.columns, null_fracs)
        if n_null / len(X_raw) <= 0.20
    ]
    dropped = len(X_raw.columns) - len(keep)
    if dropped:
        print(f'  ⚠️  Dropped {dropped} columns with >20% missing returns.')

    X = X_raw.select(keep).fill_null(0.0).to_numpy()
    print(f'  Returns matrix: {X.shape[0]} months × {X.shape[1]} assets')
    return X, dates, keep


In [ ]:
def rppca_oos_analysis(X, gamma, K, window=240, stdnorm=0,
                       variance_normalization=1, orthogonalization=1):
    """
    In-sample and rolling out-of-sample evaluation of RP-PCA.

    For each rolling window of `window` months, fits RP-PCA on the in-window
    data and records:
      - OOS alpha (pricing error) for the next month
      - OOS SDF portfolio return for the next month
      - Generalized correlation between rolling and full-sample loadings

    Parameters
    ----------
    X                      : (T, N) numpy array
    gamma                  : float, risk-premium weight
    K                      : int, number of factors
    window                 : int, rolling window length in months
    stdnorm                : int, passed through to rppca()
    variance_normalization : int, passed through to rppca()
    orthogonalization      : int, passed through to rppca()

    Returns
    -------
    dict with keys:
        'IS_Results'   : (3, K) array — [Sharpe ratios; RMSEs; var% unexplained]
        'OOS_Results'  : (3, K) array — same metrics out-of-sample
        'Corr_Final'   : (T-window, K) generalized correlations over time
        'OOS_Returns'  : (T-window, K) monthly OOS SDF portfolio returns
        'Full_IS_Model': output of rppca() on the full sample (use for weights)
    """
    if hasattr(X, 'to_numpy'):
        X = X.to_numpy()
    T, N = X.shape
    T_oos = T - window

    # ── Full in-sample model (used for IS metrics and final weights) ──
    is_model = rppca(X, gamma, K, stdnorm, variance_normalization, orthogonalization)

    sr_is, rmse_is, var_is = [], [], []
    for k in range(1, K + 1):
        alpha_k = is_model['alpha'][:, k - 1]
        resid_k = is_model['residual'][k]
        sdf_k   = is_model['SDF'][:, k - 1]
        rmse_is.append(np.sqrt(np.mean(alpha_k ** 2)))
        var_is.append(
            np.trace(np.cov(resid_k, rowvar=False))
            / np.trace(np.cov(X, rowvar=False)) * 100
        )
        sr_is.append(sdf_k.mean() / sdf_k.std())

    # ── Rolling OOS loop ─────────────────────────────────────────────
    alpha_oos      = [np.zeros((T_oos, N)) for _ in range(K)]
    oos_returns    = np.zeros((T_oos, K))   # SDF portfolio returns each month
    corr_final     = np.zeros((T_oos, K))
    lambda_prev    = None
    L_full         = is_model['loadings']

    for t in tqdm.trange(T_oos, desc='Rolling OOS'):
        X_win  = X[t: t + window, :]
        X_next = X[t + window, :].reshape(1, -1)

        res_t  = rppca(X_win, gamma, K, stdnorm, variance_normalization, orthogonalization)
        L_win  = res_t['loadings'].copy()

        # Align sign of loadings to previous window for consistency
        if lambda_prev is not None:
            signs = np.sign(np.diag(L_win.T @ lambda_prev))
            L_win = L_win @ np.diag(signs)
        lambda_prev = L_win

        for k_idx in range(K):
            k  = k_idx + 1
            Lk = L_win[:, :k]
            LkTLk_inv = linalg.inv(Lk.T @ Lk)

            # OOS pricing error
            proj = Lk @ LkTLk_inv @ Lk.T
            alpha_oos[k_idx][t, :] = (X_next @ (np.eye(N) - proj)).ravel()

            # OOS SDF portfolio return
            Fk    = X_win @ Lk @ LkTLk_inv
            mu_f  = Fk.mean(axis=0)
            cov_f = np.atleast_2d(np.cov(Fk, rowvar=False))
            w_sdf = linalg.solve(cov_f, mu_f)
            oos_returns[t, k_idx] = float(X_next @ Lk @ LkTLk_inv @ w_sdf)

        # Generalized correlation: rolling vs full-sample loadings (all K factors)
        M = (linalg.inv(L_win.T @ L_win) @ L_win.T @ L_full
             @ linalg.inv(L_full.T @ L_full) @ L_full.T @ L_win)
        corr_final[t, :] = np.sort(np.sqrt(np.abs(linalg.eigvals(M))))[::-1]

    # ── OOS metrics ──────────────────────────────────────────────────
    sr_oos, rmse_oos, var_oos = [], [], []
    X_oos = X[window:, :]
    for k_idx in range(K):
        mean_alpha = alpha_oos[k_idx].mean(axis=0)
        rmse_oos.append(np.sqrt(np.mean(mean_alpha ** 2)))
        var_oos.append(
            np.trace(np.cov(alpha_oos[k_idx], rowvar=False))
            / np.trace(np.cov(X_oos, rowvar=False)) * 100
        )
        r = oos_returns[:, k_idx]
        sr_oos.append(r.mean() / r.std())

    return {
        'IS_Results':   np.array([sr_is,   rmse_is,   var_is]),
        'OOS_Results':  np.array([sr_oos,  rmse_oos,  var_oos]),
        'Corr_Final':   corr_final,
        'OOS_Returns':  oos_returns,
        'Full_IS_Model': is_model,
    }


In [ ]:
# ── Step 1: Prepare the returns matrix ──────────────────────────
print('Preparing data...')
X, dates, asset_cols = prepare_data_for_rppca(portfolios, selected_signals)

# ── Step 2: Choose K ─────────────────────────────────────────────
# The eigenvalue-ratio test always runs as a background diagnostic;
# it determines K only when USE_AUTO_K = True.
_tmp = rppca(X, gamma=GAMMA, K=K_MAX, stdnorm=STD_NORM,
             variance_normalization=VAR_NORM, orthogonalization=ORTHOGONALIZE)
K_auto = select_k(_tmp['eigenvalues'], X.shape, k_max=K_MAX)
K = K_auto if USE_AUTO_K else K_MANUAL
print(f'\nEigenvalue-ratio diagnostic suggests K = {K_auto}  |  '
      f'using K = {K} ({"auto" if USE_AUTO_K else "manual, K_MANUAL"})')
plot_scree(_tmp['eigenvalues'], K, X.shape)
del _tmp

# ── Step 3: Full OOS analysis ────────────────────────────────────
print(f'\nRunning RP-PCA + rolling OOS analysis '
      f'(K={K}, gamma={GAMMA}, window={OOS_WINDOW} months)...')
print('  This will take a few minutes. Progress bar below:')

results = rppca_oos_analysis(
    X, gamma=GAMMA, K=K, window=OOS_WINDOW,
    stdnorm=STD_NORM, variance_normalization=VAR_NORM,
    orthogonalization=ORTHOGONALIZE,
)
model = results['Full_IS_Model']  # use this for portfolio construction
print('\n✅ Analysis complete.')


In [ ]:
# ── Sharpe Ratio & Pricing Error Table ──────────────────────────
is_res  = results['IS_Results']   # shape (3, K): [SR, RMSE, Var%]
oos_res = results['OOS_Results']

print(f'RP-PCA Results  (gamma={GAMMA}, K={K}, OOS window={OOS_WINDOW} months)\n')
header = f'{"Factors":>8}  {"IS SR":>8}  {"OOS SR":>8}  '\
         f'{"IS RMSE":>9}  {"OOS RMSE":>9}  {"IS Var%":>8}  {"OOS Var%":>9}'
print(header)
print('-' * len(header))
for k in range(K):
    print(f'{k+1:>8}  {is_res[0,k]*np.sqrt(12):>8.3f}  {oos_res[0,k]*np.sqrt(12):>8.3f}  '
          f'{is_res[1,k]:>9.4f}  {oos_res[1,k]:>9.4f}  '
          f'{is_res[2,k]:>8.2f}  {oos_res[2,k]:>9.2f}')

print(f'\nNote: SR = Sharpe ratio annualised from monthly (multiplied by sqrt(12)); '
      f'RMSE = monthly root mean squared pricing error; Var% = unexplained variance.')

# ── Cumulative SDF Portfolio Return ──────────────────────────────
oos_returns = results['OOS_Returns']  # (T_oos, K)
n_oos = oos_returns.shape[0]
oos_dates = dates[OOS_WINDOW:].to_list()

fig, ax = plt.subplots(figsize=(12, 4))
for k in range(K):
    cumret = np.cumprod(1 + oos_returns[:, k]) - 1
    ax.plot(oos_dates, cumret, label=f'K={k+1} factors')
ax.set_title('Cumulative OOS SDF Portfolio Returns')
ax.set_xlabel('Date'); ax.set_ylabel('Cumulative Return')
ax.legend(loc='upper left', ncol=3); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# ── Generalized Correlation Over Time ────────────────────────────
fig, ax = plt.subplots(figsize=(12, 3))
for k in range(K):
    ax.plot(oos_dates, results['Corr_Final'][:, k], label=f'Factor {k+1}')
ax.set_title('Stability of Loadings: Generalized Correlation (Rolling vs Full Sample)')
ax.set_xlabel('Date'); ax.set_ylabel('Correlation')
ax.set_ylim(0, 1.05); ax.legend(ncol=3); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


---
# Section 4 — Portfolio Weights → Individual Stock Weights

Maps the RP-PCA SDF weights (which live in the space of characteristic-sorted decile portfolios) back to individual NYSE/NASDAQ stocks using:
1. A permno → ticker lookup built from public crosswalk files
2. Market capitalisation data from yfinance (cached to Drive)
3. Decile re-assignment using each stock's current characteristic value
4. Exchange membership and recency filters

> **Note:** Cells 4.2 (build investable universe) and 4.3 (load saved universe) are alternatives — run **one or the other**, not both. Run 4.2 only once; thereafter load the cached result with 4.3.

In [ ]:
# ── Build permno → ticker crosswalk ─────────────────────────────
# Source: Wenzhi Ding's standardised security code files (public GitHub)
print('Building permno → ticker map...')
permno_gvkey = pl.read_parquet(
    'https://github.com/Wenzhi-Ding/Std_Security_Code/blob/main/other/gvkey_permco_permno.pq?raw=true'
).filter(
    (pl.col('LINKTYPE') == 'LU') & pl.col('LPERMNO').is_not_null()
).select(['LPERMNO', 'GVKEY']).rename({'LPERMNO': 'permno'}).with_columns(
    pl.col('permno').cast(pl.Int64)
).unique('permno', keep='first')

gvkey_isin = pl.read_parquet(
    'https://github.com/Wenzhi-Ding/Std_Security_Code/blob/main/isin/gvkey.pq?raw=true'
).filter(
    pl.col('gvkey').is_not_null() & pl.col('isin').is_not_null()
).select(['gvkey', 'isin']).with_columns(
    pl.col('gvkey').cast(pl.Int64)
).unique('isin', keep='first')

isin_ticker = pl.read_parquet(
    'https://github.com/Wenzhi-Ding/Std_Security_Code/blob/main/isin/ticker.pq?raw=true'
).filter(
    pl.col('isin').is_not_null() & pl.col('ticker').is_not_null()
).select(['isin', 'ticker']).unique('isin', keep='first')

permno_ticker_map = (
    permno_gvkey
    .join(gvkey_isin, left_on='GVKEY', right_on='gvkey', how='left')
    .join(isin_ticker, on='isin', how='left')
    .select(['permno', 'ticker'])
)
print(f'  Map covers {permno_ticker_map.filter(pl.col("ticker").is_not_null()).height} permnos.')


### 4.2  Build Investable Universe (run once; skip to 4.3 if already cached)

Fetches current market caps from yfinance and joins them to the most recent
firm characteristics. Results are saved to Drive.

In [ ]:
def fetch_market_caps(tickers, batch_size=MKT_CAP_BATCH, max_retries=3, sleep_sec=5):
    """
    Fetch current market capitalisation for each ticker via yfinance.

    Uses batched requests with retry logic to avoid rate-limit errors.
    Returns a dict {ticker: market_cap_usd}.
    """
    mkt_cap_map = {}
    batches = [tickers[i: i + batch_size] for i in range(0, len(tickers), batch_size)]

    for batch in tqdm.tqdm(batches, desc='Fetching market caps'):
        for attempt in range(max_retries):
            try:
                obj = yf.Tickers(batch)
                for t in batch:
                    try:
                        mc = obj.tickers[t].fast_info.market_cap
                        if mc and mc > 0:
                            mkt_cap_map[t] = float(mc)
                    except Exception:
                        pass
                del obj; gc.collect()
                time.sleep(sleep_sec)
                break
            except Exception as e:
                if attempt < max_retries - 1:
                    time.sleep(sleep_sec * 3)
                else:
                    print(f'  ⚠️  Batch failed after {max_retries} attempts: {e}')
    return mkt_cap_map


# ── Get the most recent characteristic row per stock ─────────────
# Stream in chunks to stay within Colab memory limits.
print('Building investable universe (streaming)...')
chunk_size = 1_000
all_permnos = predictors.select('permno').unique().collect()['permno'].to_list()
chunks = []

for i in tqdm.trange(0, len(all_permnos), chunk_size, desc='Processing permnos'):
    chunk_ids = all_permnos[i: i + chunk_size]
    chunk = (
        predictors
        .filter(pl.col('permno').is_in(chunk_ids))
        .filter(pl.col('yyyymm') > MIN_DATE_YYYYMM)
        .sort('permno', 'yyyymm')
        .group_by('permno', maintain_order=True)
        # Forward-fill characteristics up to 18 months, then take the last row
        .agg(pl.all().forward_fill(limit=18).last())
        .join(
            permno_ticker_map.lazy().select(['permno', 'ticker']),
            on='permno', how='inner'
        )
        .collect(engine='streaming')
    )
    chunks.append(chunk)

# Combine chunks, deduplicate, attach market caps
base_universe = (
    pl.concat(chunks)
    .group_by('ticker').agg(pl.all().last())  # resolve cross-chunk duplicates
)

# Fetch market caps
unique_tickers = base_universe.get_column('ticker').to_list()
print(f'\nFetching market caps for {len(unique_tickers)} tickers...')
mkt_cap_map = fetch_market_caps(unique_tickers)
print(f'  Got caps for {len(mkt_cap_map)} / {len(unique_tickers)} tickers.')

# Save cap map to Drive
with open(MKT_CAP_CACHE, 'w') as f:
    json.dump(mkt_cap_map, f)
print(f'  Saved market cap cache → {MKT_CAP_CACHE}')

# Filter to stocks with known market cap
final_investable = base_universe.with_columns(
    pl.col('ticker').replace_strict(mkt_cap_map, default=None).alias('marketcap')
).filter(pl.col('marketcap').is_not_null())

# Save to Drive
save_path = f'{DRIVE_DATA_DIR}/final_investable.parquet'
final_investable.write_parquet(save_path, compression='zstd')
print(f'\n✅ Saved investable universe ({final_investable.height} stocks) → {save_path}')


### 4.3  Load Saved Investable Universe (skip 4.2 if already cached)

In [ ]:
with open(MKT_CAP_CACHE, 'r') as f:
    mkt_cap_map = json.load(f)

final_investable = pl.read_parquet(f'{DRIVE_DATA_DIR}/final_investable.parquet')
print(f'Loaded {final_investable.height} stocks from cache.')


### 4.4  Apply Exchange and Recency Filters

Restricts the universe to NASDAQ and NYSE stocks that traded within the last `STALENESS_DAYS` days.

In [ ]:
from datetime import datetime, timedelta

# ── Official NASDAQ / NYSE ticker lists ──────────────────────────
print('Fetching official NASDAQ/NYSE ticker lists...')
nasdaq_df = pl.read_csv(
    'https://www.nasdaqtrader.com/dynamic/symdir/nasdaqlisted.txt',
    separator='|'
).head(-1)  # last row is a timestamp
other_df = pl.read_csv(
    'https://www.nasdaqtrader.com/dynamic/symdir/otherlisted.txt',
    separator='|'
).head(-1)

nasdaq_syms = set(nasdaq_df['Symbol'].to_list())
nyse_syms   = set(other_df.filter(pl.col('Exchange') == 'N')['ACT Symbol'].to_list())
prominent   = nasdaq_syms | nyse_syms
print(f'  NASDAQ: {len(nasdaq_syms):,} | NYSE: {len(nyse_syms):,} tickers')

# ── Recency check via yfinance ───────────────────────────────────
tickers_to_check = final_investable['ticker'].to_list()
print(f'\nChecking {len(tickers_to_check)} tickers for staleness and exchange...')

price_data = yf.download(
    tickers_to_check, period='1mo', group_by='ticker', progress=True, auto_adjust=True
)

# Extract close prices; handle single-ticker edge case
try:
    close = price_data.xs('Close', level=1, axis=1)
except KeyError:
    close = price_data['Close'] if 'Close' in price_data else price_data

cutoff = datetime.now() - timedelta(days=STALENESS_DAYS)
active = []
for t in tickers_to_check:
    if t not in prominent:
        continue
    if t not in close.columns:
        continue
    series = close[t].dropna()
    if series.empty:
        continue
    if series.index[-1].to_pydatetime().replace(tzinfo=None) >= cutoff:
        active.append(t)

final_investable = final_investable.filter(pl.col('ticker').is_in(active))
print(f'\n✅ Final investable universe: {final_investable.height} stocks '
      f'(NASDAQ/NYSE, active within {STALENESS_DAYS} days)')


### 4.5  Map SDF Weights to Individual Stocks

The RP-PCA SDF weights are defined over characteristic-sorted decile portfolios. To convert them to individual stock weights, we:
1. For each test asset (e.g. `Accruals_10`), identify which decile each stock falls into using its current characteristic value.
2. Distribute the portfolio weight proportionally by market cap within that decile.
3. Sum contributions across all factors and all characteristic-decile combinations.

**Note:** `asset_cols` (returned by `prepare_data_for_rppca`) determines the exact mapping between column index and characteristic/decile — this is the correct way to avoid index misalignment bugs.

In [ ]:
# Use SDF weights for the selected K (full-factor SDF)
portfolio_weights = model['SDFweightsassets'][K]  # shape: (N_assets,)

# Convert to pandas for row-level iteration
inv_pd = final_investable.to_pandas()
exclude_cols = {'permno', 'yyyymm', 'ticker', 'marketcap'}
char_columns = [c for c in inv_pd.columns if c not in exclude_cols]

stock_weights: dict[str, float] = {}
skipped = 0

for port_idx, p_weight in enumerate(portfolio_weights):
    # Recover which characteristic and which decile this portfolio weight belongs to
    # asset_cols has entries like 'Accruals_01', 'Accruals_10', 'AbnormalAccruals_01', ...
    if port_idx >= len(asset_cols):
        break
    asset_id = asset_cols[port_idx]

    # Parse: everything before the last underscore is the signal name
    try:
        char_name, decile_str = asset_id.rsplit('_', 1)
        decile_rank = int(decile_str)  # 1 or 10 (or 1-10 if DECILE_MODE='all')
    except (ValueError, AttributeError):
        skipped += 1; continue

    if char_name not in inv_pd.columns:
        skipped += 1; continue

    char_vals = inv_pd[char_name]

    # Assign stocks to the relevant decile bucket
    if DECILE_MODE == 'extreme':
        q_low  = char_vals.quantile(0.10)
        q_high = char_vals.quantile(0.90)
        if decile_rank == 1:
            mask = char_vals <= q_low
        else:  # decile 10
            mask = char_vals >= q_high
        decile_stocks = inv_pd[mask]
    else:
        # All-decile mode: use qcut (0-indexed labels 0..9)
        try:
            labels = pd.qcut(char_vals, 10, labels=False, duplicates='drop')
        except ValueError:
            skipped += 1; continue
        if labels.nunique() < decile_rank:
            skipped += 1; continue
        decile_stocks = inv_pd[labels == (decile_rank - 1)]

    if decile_stocks.empty:
        skipped += 1; continue

    total_mktcap = decile_stocks['marketcap'].sum()
    if total_mktcap <= 0:
        skipped += 1; continue

    for _, row in decile_stocks.iterrows():
        w = p_weight * (row['marketcap'] / total_mktcap)
        stock_weights[row['ticker']] = stock_weights.get(row['ticker'], 0.0) + w

if skipped:
    print(f'  ℹ️  Skipped {skipped} portfolio slots (missing characteristic data).')

investable_df = pd.DataFrame.from_dict(
    stock_weights, orient='index', columns=['weight']
).rename_axis('ticker').sort_values('weight', ascending=False)

print(f'✅ Portfolio built: {len(investable_df)} stocks')


In [ ]:
long_mask  = investable_df['weight'] > 0
short_mask = investable_df['weight'] < 0

print('═' * 45)
print('  FINAL PORTFOLIO SUMMARY')
print('═' * 45)
print(f'  Total stocks       : {len(investable_df):>6}')
print(f'  Long positions     : {long_mask.sum():>6}')
print(f'  Short positions    : {short_mask.sum():>6}')
print(f'  Gross exposure     : {investable_df["weight"].abs().sum():>9.4f}')
print(f'  Net exposure       : {investable_df["weight"].sum():>9.4f}')
print(f'  Long weight sum    : {investable_df.loc[long_mask, "weight"].sum():>9.4f}')
print(f'  Short weight sum   : {investable_df.loc[short_mask, "weight"].sum():>9.4f}')
print('═' * 45)
print('\nTop 15 long positions:')
print(investable_df.head(15).to_string())
print('\nTop 15 short positions:')
print(investable_df[short_mask].tail(15).to_string())
